# 10 — Entrenamiento Final con Features del GA

Usa los cromosomas óptimos encontrados en `09_metodos_evolutivos.ipynb` para entrenar los modelos finales con los datos completos y comparar contra el baseline (`08_experimentos_final.ipynb`).

| Etapa | Descripción |
|---|---|
| Carga | Lee `outputs/ga_resultados.json` con los cromosomas óptimos |
| Entrenamiento | Modelos completos con las features seleccionadas (epochs=120, patience=10) |
| Evaluación | Métricas en test set para regresión y clasificación |
| Comparativa | GA-REG y GA-CLF vs baseline (todas las features) |


In [ ]:
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    f1_score, roc_auc_score, precision_recall_curve,
    confusion_matrix, ConfusionMatrixDisplay, classification_report, roc_curve
)

warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f'GPU: {len(gpus)} dispositivo(s)')
else:
    print('Modo CPU')

print(f'TensorFlow {tf.__version__}')

DATA_DIR     = Path('../dataset')
OUTPUTS_DIR  = Path('../outputs')
GRAFICAS_DIR = Path('../outputs/graficas')
MODELS_DIR   = Path('../models')
for d in [OUTPUTS_DIR, GRAFICAS_DIR, MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)


## 1. Cargar Cromosomas del GA

In [ ]:
with open(OUTPUTS_DIR / 'ga_resultados.json', 'r', encoding='utf-8') as f:
    ga = json.load(f)

FEATURE_NAMES  = ga['feature_names']
N_FEATURES     = len(FEATURE_NAMES)
cromosoma_reg  = ga['ga_regresion']['cromosoma']
cromosoma_clf  = ga['ga_clasificacion']['cromosoma']
features_reg   = ga['ga_regresion']['features_activas']
features_clf   = ga['ga_clasificacion']['features_activas']

FEATURES_NUM = [
    'distancia_km', 'precio_total', 'flete_total',
    'product_weight_g', 'volumen_cm3',
    'mes_compra', 'dia_semana_compra', 'hora_compra',
    'dias_estimados', 'dias_limite_envio', 'n_items', 'mismo_estado'
]
FEATURES_CAT = ['categoria_producto', 'customer_state', 'seller_state']

print('Cromosomas cargados desde outputs/ga_resultados.json')
print(f'\nGA-REG  ({sum(cromosoma_reg)}/{N_FEATURES} features):')
for f in features_reg:
    print(f'  ✓ {f}')
print(f'\nGA-CLF  ({sum(cromosoma_clf)}/{N_FEATURES} features):')
for f in features_clf:
    print(f'  ✓ {f}')


## 2. Pipeline de Datos (completo)

In [ ]:
orders    = pd.read_csv(DATA_DIR / 'olist_orders_dataset.csv', parse_dates=[
    'order_purchase_timestamp', 'order_delivered_customer_date',
    'order_estimated_delivery_date'])
items     = pd.read_csv(DATA_DIR / 'olist_order_items_dataset.csv', parse_dates=['shipping_limit_date'])
products  = pd.read_csv(DATA_DIR / 'olist_products_dataset.csv')
customers = pd.read_csv(DATA_DIR / 'olist_customers_dataset.csv')
sellers   = pd.read_csv(DATA_DIR / 'olist_sellers_dataset.csv')
geo       = pd.read_csv(DATA_DIR / 'olist_geolocation_dataset.csv')
cat_names = pd.read_csv(DATA_DIR / 'product_category_name_translation.csv')

df = orders[(orders['order_status'] == 'delivered') &
             orders['order_delivered_customer_date'].notna()].copy()
df['dias_entrega']   = (df['order_delivered_customer_date'] - df['order_purchase_timestamp']).dt.days
df['es_retraso']     = (df['order_delivered_customer_date'] > df['order_estimated_delivery_date']).astype(int)
df = df[(df['dias_entrega'] >= 0) & (df['dias_entrega'] <= 60)]
df['dias_estimados'] = (df['order_estimated_delivery_date'] - df['order_purchase_timestamp']).dt.days
df['mes_compra']        = df['order_purchase_timestamp'].dt.month
df['dia_semana_compra'] = df['order_purchase_timestamp'].dt.dayofweek
df['hora_compra']       = df['order_purchase_timestamp'].dt.hour

items_agg = items.sort_values('price', ascending=False).groupby('order_id').agg(
    precio_total=('price','sum'), flete_total=('freight_value','sum'),
    product_id=('product_id','first'), seller_id=('seller_id','first'),
    n_items=('order_item_id','count'), shipping_limit_date=('shipping_limit_date','first')
).reset_index()
df = df.merge(items_agg, on='order_id', how='left')
df['dias_limite_envio'] = (df['shipping_limit_date'] - df['order_purchase_timestamp']).dt.days.clip(lower=0)

products = products.merge(cat_names, on='product_category_name', how='left')
df = df.merge(products[['product_id','product_category_name_english',
    'product_weight_g','product_length_cm','product_height_cm','product_width_cm']],
    on='product_id', how='left')
df.rename(columns={'product_category_name_english': 'categoria_producto'}, inplace=True)
df['volumen_cm3'] = df['product_length_cm'] * df['product_height_cm'] * df['product_width_cm']
df = df.merge(sellers[['seller_id','seller_zip_code_prefix','seller_state']], on='seller_id', how='left')
df = df.merge(customers[['customer_id','customer_zip_code_prefix','customer_state']], on='customer_id', how='left')
df['mismo_estado'] = (df['seller_state'] == df['customer_state']).astype(int)

geo_agg = geo.groupby('geolocation_zip_code_prefix').agg(
    lat=('geolocation_lat','median'), lng=('geolocation_lng','median')).reset_index()
df = df.merge(geo_agg.rename(columns={'geolocation_zip_code_prefix':'seller_zip_code_prefix',
    'lat':'seller_lat','lng':'seller_lng'}), on='seller_zip_code_prefix', how='left')
df = df.merge(geo_agg.rename(columns={'geolocation_zip_code_prefix':'customer_zip_code_prefix',
    'lat':'customer_lat','lng':'customer_lng'}), on='customer_zip_code_prefix', how='left')

def haversine(lat1, lng1, lat2, lng2):
    R = 6371.0
    lat1, lng1, lat2, lng2 = map(np.radians, [lat1, lng1, lat2, lng2])
    a = np.sin((lat2-lat1)/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin((lng2-lng1)/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

df['distancia_km'] = haversine(df['seller_lat'], df['seller_lng'],
                                df['customer_lat'], df['customer_lng'])

df_modelo = df[FEATURES_NUM + FEATURES_CAT + ['dias_entrega','es_retraso','order_purchase_timestamp']].copy()
for col in FEATURES_NUM:
    df_modelo[col] = df_modelo[col].fillna(df_modelo[col].median())
for col in FEATURES_CAT:
    df_modelo[col] = df_modelo[col].fillna('desconocido')

df_modelo = df_modelo.sort_values('order_purchase_timestamp').reset_index(drop=True)
n = len(df_modelo)
n_train, n_val = int(n * 0.80), int(n * 0.90)
train = df_modelo.iloc[:n_train].copy()
val   = df_modelo.iloc[n_train:n_val].copy()
test  = df_modelo.iloc[n_val:].copy()

scaler = StandardScaler()
train[FEATURES_NUM] = scaler.fit_transform(train[FEATURES_NUM])
val[FEATURES_NUM]   = scaler.transform(val[FEATURES_NUM])
test[FEATURES_NUM]  = scaler.transform(test[FEATURES_NUM])

encoders, cardinalidades, IDX_DESCONOCIDO = {}, {}, {}
for col in FEATURES_CAT:
    le = LabelEncoder()
    # Incluir 'desconocido' en el fit para que le.classes_ siempre lo contenga
    train_vals = list(train[col].unique())
    if 'desconocido' not in train_vals:
        train_vals.append('desconocido')
    le.fit(sorted(train_vals))
    for split in [train, val, test]:
        split[col] = split[col].apply(lambda x: x if x in le.classes_ else 'desconocido')
    train[col] = le.transform(train[col])
    val[col]   = le.transform(val[col])
    test[col]  = le.transform(test[col])
    encoders[col]        = le
    cardinalidades[col]  = len(le.classes_)
    IDX_DESCONOCIDO[col] = int(le.transform(['desconocido'])[0])

def preparar_inputs(df_split, mascara_num, mascara_cat):
    num_arr = df_split[FEATURES_NUM].values.astype('float32').copy()
    num_arr[:, ~np.array(mascara_num, dtype=bool)] = 0.0
    inputs = [num_arr]
    for i, col in enumerate(FEATURES_CAT):
        arr = df_split[col].values.astype('int32').copy()
        if not mascara_cat[i]:
            arr[:] = IDX_DESCONOCIDO[col]
        inputs.append(arr)
    return inputs

mascara_num_reg = cromosoma_reg[:len(FEATURES_NUM)]
mascara_cat_reg = cromosoma_reg[len(FEATURES_NUM):]
mascara_num_clf = cromosoma_clf[:len(FEATURES_NUM)]
mascara_cat_clf = cromosoma_clf[len(FEATURES_NUM):]

X_train_reg = preparar_inputs(train, mascara_num_reg, mascara_cat_reg)
X_val_reg   = preparar_inputs(val,   mascara_num_reg, mascara_cat_reg)
X_test_reg  = preparar_inputs(test,  mascara_num_reg, mascara_cat_reg)

X_train_clf = preparar_inputs(train, mascara_num_clf, mascara_cat_clf)
X_val_clf   = preparar_inputs(val,   mascara_num_clf, mascara_cat_clf)
X_test_clf  = preparar_inputs(test,  mascara_num_clf, mascara_cat_clf)

y_train_reg = train['dias_entrega'].values.astype('float32')
y_val_reg   = val['dias_entrega'].values.astype('float32')
y_test_reg  = test['dias_entrega'].values.astype('float32')
y_train_clf = train['es_retraso'].values.astype('float32')
y_val_clf   = val['es_retraso'].values.astype('float32')
y_test_clf  = test['es_retraso'].values.astype('float32')

print(f'Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}')


In [ ]:
# ── 3. Funciones Auxiliares (NN + Focal Loss) ──────────────────────────────
# Misma arquitectura del notebook 08: [256,128,64] | Dropout(0.3,0.2) | Focal Loss γ=2 α=0.75

CAPAS   = [256, 128, 64]
DROPOUT = (0.3, 0.2)
EPOCHS  = 120
BATCH   = 256

def construir_modelo(capas, dropout, activacion_salida, nombre_salida):
    inp_num = keras.Input(shape=(len(FEATURES_NUM),), name='input_numerico')
    emb_inputs, emb_outputs = [], []
    for col, card in cardinalidades.items():
        dim = min(50, (card // 2) + 1)
        inp = keras.Input(shape=(1,), name=f'input_{col}', dtype='int32')
        emb = layers.Embedding(input_dim=card, output_dim=dim, name=f'emb_{col}')(inp)
        emb = layers.Flatten()(emb)
        emb_inputs.append(inp)
        emb_outputs.append(emb)
    x = layers.Concatenate()([inp_num] + emb_outputs)
    for i, u in enumerate(capas):
        x = layers.Dense(u, activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(dropout[0] if i == 0 else dropout[1])(x)
    out = layers.Dense(1, activation=activacion_salida, name=nombre_salida)(x)
    return keras.Model(inputs=[inp_num] + emb_inputs, outputs=out)


def focal_loss(gamma=2.0, alpha=0.75):
    def loss(y_true, y_pred):
        y_pred  = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        bce     = -y_true * tf.math.log(y_pred) - (1 - y_true) * tf.math.log(1 - y_pred)
        p_t     = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        alpha_t = y_true * alpha + (1 - y_true) * (1 - alpha)
        return tf.reduce_mean(alpha_t * tf.pow(1 - p_t, gamma) * bce)
    return loss


def umbral_optimo(modelo, X_val, y_val):
    probs = modelo.predict(X_val, verbose=0).flatten()
    prec, rec, thr = precision_recall_curve(y_val, probs)
    f1s = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-8)
    return float(thr[np.argmax(f1s)])


def callbacks_fn(monitor, mode):
    return [
        keras.callbacks.EarlyStopping(monitor=monitor, patience=10,
            restore_best_weights=True, mode=mode, verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
            patience=5, min_lr=1e-6, verbose=0)
    ]

print('Funciones definidas.')


## 4. Entrenamiento Final — Regresión (features GA-REG)

Entrenamos el modelo de regresión con las **features seleccionadas por el GA** para maximizar R².


In [ ]:
print(f"Features GA-REG ({sum(cromosoma_reg)}/15): {features_reg}")

tf.keras.backend.clear_session()
modelo_reg_me = construir_modelo(CAPAS, DROPOUT, 'linear', 'dias_entrega')
modelo_reg_me.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='mae', metrics=['mae']
)

hist_reg_me = modelo_reg_me.fit(
    X_train_reg, y_train_reg,
    validation_data=(X_val_reg, y_val_reg),
    epochs=EPOCHS, batch_size=BATCH,
    callbacks=callbacks_fn('val_mae', 'min'),
    verbose=1
)

preds_reg = modelo_reg_me.predict(X_test_reg, verbose=0).flatten()
mae_me    = float(mean_absolute_error(y_test_reg, preds_reg))
rmse_me   = float(np.sqrt(mean_squared_error(y_test_reg, preds_reg)))
r2_me     = float(r2_score(y_test_reg, preds_reg))

print(f'\n=== Regresión GA-REG ===')
print(f'  MAE  : {mae_me:.4f} días')
print(f'  RMSE : {rmse_me:.4f} días')
print(f'  R²   : {r2_me:.4f}')

MODELS_DIR.mkdir(exist_ok=True)
modelo_reg_me.save(MODELS_DIR / 'modelo_regresion_ME.keras')
print('Modelo guardado en models/modelo_regresion_ME.keras')


## 5. Entrenamiento Final — Clasificación (features GA-CLF)

Entrenamos el modelo de clasificación con las **features seleccionadas por el GA** para maximizar F1.


In [ ]:
print(f"Features GA-CLF ({sum(cromosoma_clf)}/15): {features_clf}")

tf.keras.backend.clear_session()
modelo_clf_me = construir_modelo(CAPAS, DROPOUT, 'sigmoid', 'es_retraso')
modelo_clf_me.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=focal_loss(gamma=2.0, alpha=0.75),
    metrics=[keras.metrics.AUC(name='auc')]
)

hist_clf_me = modelo_clf_me.fit(
    X_train_clf, y_train_clf,
    validation_data=(X_val_clf, y_val_clf),
    epochs=EPOCHS, batch_size=BATCH,
    callbacks=callbacks_fn('val_auc', 'max'),
    verbose=1
)

umbral_me = umbral_optimo(modelo_clf_me, X_val_clf, y_val_clf)
probs_clf  = modelo_clf_me.predict(X_test_clf, verbose=0).flatten()
preds_clf  = (probs_clf >= umbral_me).astype(int)
auc_me     = float(roc_auc_score(y_test_clf, probs_clf))
f1_me      = float(f1_score(y_test_clf, preds_clf))

print(f'\n=== Clasificación GA-CLF ===')
print(f'  AUC     : {auc_me:.4f}')
print(f'  F1      : {f1_me:.4f}')
print(f'  Umbral  : {umbral_me:.4f}')

modelo_clf_me.save(MODELS_DIR / 'modelo_clasificacion_ME.keras')
print('Modelo guardado en models/modelo_clasificacion_ME.keras')


## 6. Resultados Comparativos

Comparación entre el modelo **baseline (15 features)** del notebook 08 y los modelos con features seleccionadas por el GA.


In [ ]:
# ── Cargar métricas baseline ────────────────────────────────────────────────
with open(OUTPUTS_DIR / 'metricas_final.json', 'r') as f:
    bl = json.load(f)

mae_bl  = bl['regresion']['mae_dias']
rmse_bl = bl['regresion']['rmse_dias']
r2_bl   = bl['regresion']['r2']
auc_bl  = bl['clasificacion']['auc_roc']
f1_bl   = bl['clasificacion']['f1_score']
n_bl    = 15

# ── Tabla comparativa ────────────────────────────────────────────────────────
header = f"{'Modelo':<22} {'Features':>9} {'MAE':>8} {'RMSE':>8} {'R²':>8} {'AUC':>8} {'F1':>8}"
sep    = '-' * len(header)

def delta(new, ref, better='lower'):
    d = (new - ref) / abs(ref + 1e-8) * 100
    sign = '+' if d > 0 else ''
    arrow = '▼' if (better == 'lower' and d < 0) or (better == 'higher' and d > 0) else '▲'
    return f"{sign}{d:.1f}% {arrow}"

print(header)
print(sep)
print(f"{'Baseline (08, 15 feat)':<22} {n_bl:>9} {mae_bl:>8.4f} {rmse_bl:>8.4f} {r2_bl:>8.4f} {auc_bl:>8.4f} {f1_bl:>8.4f}")
print(f"{'GA-REG (reg features)':<22} {sum(cromosoma_reg):>9} {mae_me:>8.4f} {rmse_me:>8.4f} {r2_me:>8.4f} {'—':>8} {'—':>8}   Δ MAE:{delta(mae_me,mae_bl,'lower')}  Δ R²:{delta(r2_me,r2_bl,'higher')}")
print(f"{'GA-CLF (clf features)':<22} {sum(cromosoma_clf):>9} {'—':>8} {'—':>8} {'—':>8} {auc_me:>8.4f} {f1_me:>8.4f}   Δ AUC:{delta(auc_me,auc_bl,'higher')}  Δ F1:{delta(f1_me,f1_bl,'higher')}")
print(sep)


In [ ]:
# ── Gráfico de barras comparativo ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Comparación: Baseline vs Modelos con Features GA', fontsize=14, fontweight='bold')

# Panel izquierdo: Regresión
ax = axes[0]
metrics_reg = ['MAE', 'RMSE', 'R²']
vals_bl_reg = [mae_bl, rmse_bl, r2_bl]
vals_me_reg = [mae_me, rmse_me, r2_me]
x = np.arange(len(metrics_reg))
w = 0.35
b1 = ax.bar(x - w/2, vals_bl_reg, w, label='Baseline (15 feat)', color='steelblue', alpha=0.85)
b2 = ax.bar(x + w/2, vals_me_reg, w, label=f'GA-REG ({sum(cromosoma_reg)} feat)', color='darkorange', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(metrics_reg)
ax.set_title('Regresión (días de retraso)')
ax.set_ylabel('Valor')
ax.legend()
for bar in [*b1, *b2]:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)

# Panel derecho: Clasificación
ax = axes[1]
metrics_clf = ['AUC', 'F1']
vals_bl_clf = [auc_bl, f1_bl]
vals_me_clf = [auc_me, f1_me]
x = np.arange(len(metrics_clf))
b1 = ax.bar(x - w/2, vals_bl_clf, w, label='Baseline (15 feat)', color='steelblue', alpha=0.85)
b2 = ax.bar(x + w/2, vals_me_clf, w, label=f'GA-CLF ({sum(cromosoma_clf)} feat)', color='seagreen', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(metrics_clf)
ax.set_title('Clasificación (retraso sí/no)')
ax.set_ylabel('Valor')
ax.set_ylim(0, 1.1)
ax.legend()
for bar in [*b1, *b2]:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'comparativo_ME.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico guardado en outputs/comparativo_ME.png')


In [ ]:
# ── Mapa de features seleccionadas ──────────────────────────────────────────
all_features = FEATURES_NUM + FEATURES_CAT
n_feat = len(all_features)

mask_reg_arr = np.array(cromosoma_reg, dtype=float)
mask_clf_arr = np.array(cromosoma_clf, dtype=float)
# Encode: 2=both, 1=single, 0=none
combined = mask_reg_arr + mask_clf_arr  # 0, 1 or 2

fig, ax = plt.subplots(figsize=(8, 0.5 * n_feat + 2))
colors_map = {0: '#d9d9d9', 1: '#fdae6b', 2: '#2ca02c'}
labels_map = {0: 'Ninguno', 1: 'Solo uno', 2: 'Ambos GA'}

y_pos = np.arange(n_feat)
for i, feat in enumerate(all_features):
    val = int(combined[i])
    reg_sel = bool(mask_reg_arr[i])
    clf_sel = bool(mask_clf_arr[i])
    color = colors_map[val]
    ax.barh(i, 1, color=color, edgecolor='white', height=0.7)
    label_txt = (
        '✓ REG + ✓ CLF' if val == 2
        else ('✓ REG' if reg_sel else '✓ CLF') if val == 1
        else '✗'
    )
    ax.text(0.5, i, label_txt, ha='center', va='center', fontsize=9,
            color='white' if val == 2 else ('#333' if val == 0 else 'white'))

ax.set_yticks(y_pos)
ax.set_yticklabels(all_features)
ax.set_xticks([])
ax.set_title('Features seleccionadas por GA-REG y GA-CLF', fontsize=13, fontweight='bold')
ax.invert_yaxis()

from matplotlib.patches import Patch
legend_elems = [
    Patch(facecolor=colors_map[2], label='Ambos GA'),
    Patch(facecolor=colors_map[1], label='Solo uno'),
    Patch(facecolor=colors_map[0], label='No seleccionada'),
]
ax.legend(handles=legend_elems, loc='lower right')
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'features_ME.png', dpi=150, bbox_inches='tight')
plt.show()
print('Mapa de features guardado en outputs/features_ME.png')


In [ ]:
# ── Curvas de aprendizaje ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Curvas de Aprendizaje — Modelos ME', fontsize=13, fontweight='bold')

# Regresión
ax = axes[0]
ax.plot(hist_reg_me.history['mae'],     label='Train MAE', color='steelblue')
ax.plot(hist_reg_me.history['val_mae'], label='Val MAE',   color='darkorange', linestyle='--')
best_ep = int(np.argmin(hist_reg_me.history['val_mae']))
ax.axvline(best_ep, color='gray', linestyle=':', linewidth=1, label=f'Best epoch {best_ep+1}')
ax.set_title(f'GA-REG ({sum(cromosoma_reg)} features)')
ax.set_xlabel('Época'); ax.set_ylabel('MAE (días)')
ax.legend(); ax.grid(alpha=0.3)

# Clasificación
ax = axes[1]
ax.plot(hist_clf_me.history['auc'],     label='Train AUC', color='seagreen')
ax.plot(hist_clf_me.history['val_auc'], label='Val AUC',   color='darkorange', linestyle='--')
best_ep_c = int(np.argmax(hist_clf_me.history['val_auc']))
ax.axvline(best_ep_c, color='gray', linestyle=':', linewidth=1, label=f'Best epoch {best_ep_c+1}')
ax.set_title(f'GA-CLF ({sum(cromosoma_clf)} features)')
ax.set_xlabel('Época'); ax.set_ylabel('AUC')
ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'curvas_ME.png', dpi=150, bbox_inches='tight')
plt.show()
print('Curvas de aprendizaje guardadas en outputs/curvas_ME.png')


## 7. Guardar Métricas Finales


In [ ]:
metricas_ME = {
    "regresion_ME": {
        "mae":        mae_me,
        "rmse":       rmse_me,
        "r2":         r2_me,
        "n_features": int(sum(cromosoma_reg)),
        "features_activas": features_reg
    },
    "clasificacion_ME": {
        "auc":        auc_me,
        "f1":         f1_me,
        "umbral":     float(umbral_me),
        "n_features": int(sum(cromosoma_clf)),
        "features_activas": features_clf
    },
    "baseline_ref": {
        "mae":        mae_bl,
        "rmse":       rmse_bl,
        "r2":         r2_bl,
        "auc":        auc_bl,
        "f1":         f1_bl,
        "n_features": 15
    },
    "cromosomas": {
        "reg": cromosoma_reg,
        "clf": cromosoma_clf
    }
}

with open(OUTPUTS_DIR / 'metricas_ME.json', 'w') as f:
    json.dump(metricas_ME, f, indent=2)

print('✔ metricas_ME.json guardado')
print('\n──── Resumen Final ────')
print(f"  Regresión   — MAE {mae_me:.4f}  R² {r2_me:.4f}  ({sum(cromosoma_reg)}/15 features)")
print(f"  Clasificación — AUC {auc_me:.4f}  F1 {f1_me:.4f}  ({sum(cromosoma_clf)}/15 features)")
print(f"\n  Δ vs Baseline (notebook 08):")
print(f"    MAE  {mae_bl:.4f} → {mae_me:.4f}  ({(mae_me - mae_bl)/mae_bl*100:+.1f}%)")
print(f"    R²   {r2_bl:.4f} → {r2_me:.4f}  ({(r2_me  - r2_bl )/max(abs(r2_bl),1e-8)*100:+.1f}%)")
print(f"    AUC  {auc_bl:.4f} → {auc_me:.4f}  ({(auc_me - auc_bl)/auc_bl*100:+.1f}%)")
print(f"    F1   {f1_bl:.4f} → {f1_me:.4f}  ({(f1_me  - f1_bl )/max(f1_bl,1e-8)*100:+.1f}%)")
